# Backpropagation — How Gradients Flow Back

Backprop is the algorithm that computes gradients for every weight in the network.
It applies the **chain rule** of calculus repeatedly from the loss backward through each layer.

**Chain rule:** if `z = f(y)` and `y = g(x)`, then `dz/dx = dz/dy × dy/dx`

In a neural network:
```
Loss → Layer N → Layer N-1 → ... → Layer 1 → Input
```
Gradient flows backward through this chain, each layer multiplying by its local gradient.

Autograd does this automatically. This notebook shows what it's actually computing.

In [ ]:
import torch
import torch.nn as nn

# --- Manual backprop through a single linear layer ---
# Forward:  y = Wx + b,  loss = (y - target)^2
# Backward: dL/dW = dL/dy * dy/dW = 2(y - target) * x
#           dL/db = dL/dy * dy/db = 2(y - target) * 1

torch.manual_seed(0)
x      = torch.tensor([[1.0, 2.0, 3.0]])
target = torch.tensor([[5.0]])

W = torch.randn(1, 3, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Forward
y    = x @ W.T + b
loss = (y - target)**2

print(f'y:    {y.item():.4f}')
print(f'loss: {loss.item():.4f}')

# Autograd backward
loss.backward()

print(f'\ndL/dW (autograd): {W.grad}')
print(f'dL/db (autograd): {b.grad}')

# Manual calculation for verification
err         = 2 * (y - target)           # dL/dy
dL_dW_manual = err * x                    # dy/dW = x
dL_db_manual = err

print(f'\ndL/dW (manual):   {dL_dW_manual.detach()}')
print(f'dL/db (manual):   {dL_db_manual.item():.4f}')
print(f'Match: {torch.allclose(W.grad, dL_dW_manual)}')

## Chain rule through two layers

```
x → [Linear1] → h → [ReLU] → a → [Linear2] → y → [Loss] → L

dL/dW1 = dL/dy × dy/da × da/dh × dh/dW1
                           ↑
                    This is the chain rule
                    Each × is one layer's local gradient
```

In [ ]:
torch.manual_seed(1)

x      = torch.randn(1, 4)
target = torch.tensor([[2.0]])

W1 = torch.randn(4, 4, requires_grad=True)
b1 = torch.zeros(4, requires_grad=True)
W2 = torch.randn(1, 4, requires_grad=True)
b2 = torch.zeros(1, requires_grad=True)

# Forward pass
h    = x @ W1 + b1         # linear 1
a    = torch.relu(h)        # activation
y    = a @ W2.T + b2        # linear 2
loss = ((y - target)**2).mean()

print('Forward pass done. Loss:', loss.item())

# Backward — autograd computes everything
loss.backward()

print('\nGradients computed:')
print('W1.grad shape:', W1.grad.shape)
print('W2.grad shape:', W2.grad.shape)
print('b1.grad shape:', b1.grad.shape)

## Vanishing gradients — why deep networks are hard to train

If gradients get multiplied by small numbers at each layer, they shrink exponentially.
By the time you reach early layers, gradients are nearly zero → weights don't update → learning stops.

In [ ]:
# Sigmoid activation causes vanishing gradients
# sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x)) — max value is 0.25 at x=0
# After 10 layers: 0.25^10 ≈ 0.000001 — gradient nearly zero

class DeepSigmoidNet(nn.Module):
    def __init__(self, depth):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(32, 32), nn.Sigmoid()]
        layers.append(nn.Linear(32, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class DeepReLUNet(nn.Module):
    def __init__(self, depth):
        super().__init__()
        layers = []
        for _ in range(depth):
            layers += [nn.Linear(32, 32), nn.ReLU()]
        layers.append(nn.Linear(32, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
x      = torch.randn(8, 32)
target = torch.randn(8, 1)
loss_fn = nn.MSELoss()

for Net, name in [(DeepSigmoidNet, 'Sigmoid'), (DeepReLUNet, 'ReLU   ')]:
    model = Net(depth=10)
    loss  = loss_fn(model(x), target)
    loss.backward()

    # Gradient magnitude at the first layer
    first_layer_grad = model.net[0].weight.grad.abs().mean().item()
    print(f'{name} — first layer avg gradient magnitude: {first_layer_grad:.8f}')

## Solutions to vanishing gradients

| Problem | Solution | How it helps |
|---|---|---|
| Sigmoid/Tanh activations | ReLU, GELU, SiLU | Gradient = 1 for positive inputs (doesn't vanish) |
| Deep networks | Residual connections | Gradient flows directly through skip connections |
| Poor initialization | Xavier/Kaiming init | Keeps activations in a stable range from the start |
| LSTMs | Gating mechanism | Cell state allows gradient to flow unchanged |

In [ ]:
# Residual connection — the key insight in ResNet and Transformers
# Instead of: x → f(x)
# Do:         x → x + f(x)   ← gradient of x passes through unchanged

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return x + self.net(x)   # residual connection: gradients flow through both paths


# In a deep residual network:
deep_resnet = nn.Sequential(*[ResidualBlock(32) for _ in range(20)])

torch.manual_seed(0)
x    = torch.randn(4, 32)
out  = deep_resnet(x)
loss = out.mean()
loss.backward()

first_grad = deep_resnet[0].net[0].weight.grad.abs().mean().item()
print(f'Deep ResNet (20 blocks) first layer gradient: {first_grad:.8f}')  # healthy gradient